In [ ]:
# ==============================================================================
# Celda 1: Verificar aceleración por GPU (NVIDIA T4 en Google Colab)
# ==============================================================================
!nvidia-smi

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "⚠️ ¡GPU no detectada!\n"
        "Para acelerar el entrenamiento x20-x40 veces, activa la GPU en Colab:\n"
        "1. Menú superior: Entorno de ejecución -> Cambiar tipo de entorno de ejecución\n"
        "2. Acelerador de hardware: T4 GPU\n"
        "3. Guarda los cambios y vuelve a ejecutar esta celda."
    )

gpu_name = torch.cuda.get_device_name(0)
print(f"\n✅ GPU detectada y lista para entrenar: {gpu_name}")


In [ ]:
# ==============================================================================
# Celda 2: Montar Google Drive, instalar dependencias y extraer dataset
# ==============================================================================
import os
import shutil
import zipfile
from google.colab import drive

# 1. Montar Google Drive
drive.mount('/content/drive')

# 2. Instalar dependencias optimizadas
!pip install --quiet segmentation-models-pytorch onnx onnxruntime

DRIVE_DIR = '/content/drive/MyDrive/ClashFarmerbot_train'
ZIP_PATH = os.path.join(DRIVE_DIR, 'coc_walls_unet.zip')
LOCAL_DATA_DIR = '/content/coc_walls_unet'

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(
        f"⚠️ No se encontró el dataset en: {ZIP_PATH}\n"
        "Asegúrate de haber subido el archivo 'coc_walls_unet.zip' a tu Google Drive dentro de la carpeta 'ClashFarmerbot_train'."
    )

# 3. Copiar train_wall_unet.py si existe en Drive
drive_script = os.path.join(DRIVE_DIR, 'train_wall_unet.py')
if os.path.exists(drive_script):
    shutil.copy2(drive_script, '/content/train_wall_unet.py')
    print('✅ train_wall_unet.py copiado desde Google Drive a /content/')
elif not os.path.exists('/content/train_wall_unet.py'):
    print('⚠️ No se encontró train_wall_unet.py en Drive ni en /content. Súbelo para continuar.')

# 4. Descomprimir dataset en SSD local (/content/coc_walls_unet) para máxima velocidad I/O
print(f"📦 Descomprimiendo {ZIP_PATH} en SSD local...")
os.makedirs('/content', exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall('/content')

# Asegurar ruta canónica si el zip no contenía prefijo coc_walls_unet
if not os.path.exists(LOCAL_DATA_DIR) and os.path.exists('/content/train'):
    os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
    shutil.move('/content/train', os.path.join(LOCAL_DATA_DIR, 'train'))
    shutil.move('/content/val', os.path.join(LOCAL_DATA_DIR, 'val'))

num_train = len(os.listdir(os.path.join(LOCAL_DATA_DIR, 'train', 'images'))) if os.path.exists(os.path.join(LOCAL_DATA_DIR, 'train', 'images')) else 0
num_val = len(os.listdir(os.path.join(LOCAL_DATA_DIR, 'val', 'images'))) if os.path.exists(os.path.join(LOCAL_DATA_DIR, 'val', 'images')) else 0
print(f"✅ Dataset listo en {LOCAL_DATA_DIR}: {num_train} imágenes de train, {num_val} imágenes de val.")


In [ ]:
# ==============================================================================
# Celda 3: Entrenamiento U-Net (MobileNetV2) con GPU T4
# ==============================================================================
import os

DRIVE_DIR = '/content/drive/MyDrive/ClashFarmerbot_train'
os.makedirs(DRIVE_DIR, exist_ok=True)

ONNX_OUT = os.path.join(DRIVE_DIR, 'wall_seg.onnx')
PT_OUT = os.path.join(DRIVE_DIR, 'wall_unet_best.pt')

print('🚀 Iniciando entrenamiento U-Net (60 épocas, batch-size=8, lr=0.001)...\n')
!python /content/train_wall_unet.py \
    --data-dir /content/coc_walls_unet \
    --onnx-out {ONNX_OUT} \
    --pt-out {PT_OUT} \
    --epochs 60 \
    --batch-size 8 \
    --lr 0.001

print('\n🎉 ¡Entrenamiento finalizado con éxito!')
print(f'📦 Modelo ONNX guardado en Drive: {ONNX_OUT}')
print(f'📦 Checkpoint PyTorch guardado en Drive: {PT_OUT}')


In [ ]:
# ==============================================================================
# Celda 4: Evaluación de métricas Dice e IoU imagen por imagen
# ==============================================================================
import glob
import os
import cv2
import numpy as np
import onnxruntime as ort

ONNX_PATH = '/content/drive/MyDrive/ClashFarmerbot_train/wall_seg.onnx'
VAL_IMG_DIR = '/content/coc_walls_unet/val/images'
VAL_MASK_DIR = '/content/coc_walls_unet/val/masks'

if not os.path.exists(ONNX_PATH):
    raise FileNotFoundError(f'No se encontró el modelo entrenado en {ONNX_PATH}')

val_imgs = sorted(glob.glob(os.path.join(VAL_IMG_DIR, '*.png')))
val_masks = sorted(glob.glob(os.path.join(VAL_MASK_DIR, '*.png')))

print(f'🔍 Evaluando {len(val_imgs)} imágenes del split de validación con {ONNX_PATH}...')

# Cargar sesión ONNX (priorizando GPU CUDA, con fallback a CPU)
avail_providers = ort.get_available_providers()
providers = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if 'CUDAExecutionProvider' in avail_providers else ['CPUExecutionProvider']
sess = ort.InferenceSession(ONNX_PATH, providers=providers)
input_name = sess.get_inputs()[0].name

IMG_W, IMG_H = 768, 352
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def predict_mask(img_bgr: np.ndarray, conf_thresh: float = 0.5) -> np.ndarray:
    h_orig, w_orig = img_bgr.shape[:2]
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(rgb, (IMG_W, IMG_H), interpolation=cv2.INTER_AREA)
    norm = ((resized.astype(np.float32) / 255.0) - MEAN) / STD
    tensor = norm.transpose(2, 0, 1)[np.newaxis, ...].astype(np.float32)
    logits = sess.run(None, {input_name: tensor})[0]
    prob = 1.0 / (1.0 + np.exp(-logits[0, 0]))
    mask_bin = (prob >= conf_thresh).astype(np.uint8) * 255
    return cv2.resize(mask_bin, (w_orig, h_orig), interpolation=cv2.INTER_NEAREST)

dices, ious = [], []
results = []

for img_path, mask_path in zip(val_imgs, val_masks):
    img = cv2.imread(img_path)
    gt = (cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE) > 127).astype(np.uint8)
    pred = (predict_mask(img) > 127).astype(np.uint8)

    inter = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()
    total = pred.sum() + gt.sum()

    dice = float((2.0 * inter + 1e-6) / (total + 1e-6))
    iou = float((inter + 1e-6) / (union + 1e-6))
    dices.append(dice)
    ious.append(iou)

    stem = os.path.basename(img_path)
    is_target = '0020' in stem
    results.append((stem, dice, iou, is_target))

print('\n' + '=' * 76)
print(f"{'Imagen de Validación':<36} | {'Dice Score':<12} | {'IoU Score':<12}")
print('-' * 76)
for stem, dice, iou, is_target in results:
    hl = ' ⭐ [TARGET]' if is_target else ''
    print(f"{stem:<36} | {dice:<12.4f} | {iou:<12.4f}{hl}")

print('-' * 76)
print(f"{'PROMEDIO GLOBAL':<36} | {np.mean(dices):<12.4f} | {np.mean(ious):<12.4f}")
print('=' * 76)
